# x AI x

This is the roadmap I followed to build my own AI model from scratch, that will auto-sort my images based on the images content.

## Pre-Text:

I have no previous previous AI Development experience, but I do contain the following skills going into this project.
This Notebook will assume that all readers have atleast the same skills going into this.

Pre-Obtained Skills:

```markdown
1. `Bash` - Advanced (This Project was done on Linux OS, so similar projects completed on Linux OS should be pre-sufficent in `Bash`) 

2. `Python` - Intermediate (I'm not sufficent in `OOP` yet but sufficent everywhere else)
  - `Numpy` - Beginner (Creating Vectors and/or Arrays)
  - `Pillow` - Intermediate (Opening, Converting, Saving, and MetaTag Editing for supported media files)
  - `SciKit-Learn` - None (No previous experience whatsover)
  - `TensorFlow` - None (No previous experience whatsoever)
  - `Torch`|`PyTorch` - None (No previous experience whatsoever)
  - `Transformers` - None (No previous experience whatsoever)

3. Serialization & Database Languages:
  - `CONF`/`INI` - Advanced (Full understanding, Creating and Parsing)
  - `CSV` - Advanced (Full understanding, Creating and Parsing)
  - `JSON` - Advanced (Full understanding, Creating and Parsing)
  - `SQL` - Intermediate (Locally: Making Tables and Inserting Data. Not sufficent at anylizing existing data, Not sufficent at proper Indexing)
```

I have a blog where I post AI-Generated anime art. As such, I have a folder with a ton of unsorted images. The job of the model will be to:

```markdown
- Stage-1:
  1. Decide if the image is indeed 'Anime art' or not.
  2. Move the image to the appropriate 'yes' or 'no' folder.

- Stage-2:
  1. Decide if the image is indeed 'Anime art' or not.
    - If the image is NOT, move it to the 'no' folder, then stop here.
    - If the image IS 'Anime art', continue onto step 2.
    
  2. Decide the 'Content Group' of the image, and move the image to that groups folder. Current 'Content Groups' include:
    - 'CyberPunk'
    - 'Fantasy'
    - 'Landscape'
    - 'Nsfw'
    - 'SciFi'
```

## [IDX] Steps Index

1. [1.0] Creating Initial Dataset
  - [1.1] Pre-Sort Images for Processing
   - [1.1-1] Suggested Project Directory Schema
   - [1.1-2] Valid Image Encoding Formats
   - [1.1-3] Pre-Sorting Images into the same Encoding Format
   - [1.1-4] 
  - /1.2/ Processes Sorted Images into usable numpy arrays
   - /1.2.1/ Open and Resize Images to Input Target Size
   - /1.2.2/ Convert image to raw RGB data format
   - /1.2.3/ Create, Shape, and Fill a numpy vector/array
   - /1.2.4/ Flatten the numpy array into an 'AI Usable' vector

## [1.0] Creating and Processing Initial Dataset

To get started. We need to create our inital datasets that we'll use to pre-train our model.
So first we must put all our images into appropriate directories.
I suggest creating an `images` folder in our projects root directory.
And creating a `RAW` sub-directory to hold all our images.

### [1.1] Pre-Sorting Images for Processing

AI expects incoming data to always be of the same shape, and size.
This posses the first hurdle for Computer-Vision projects as images with a different resolution will produce vastly different vectors / arrays.
For ease, we will start by making sure each image is the same encoding format. I will be using the `PNG` encoding.
So our first Py-Script should sort the `*/images/RAW` directory, removing/converting all images to the target encoding.
Then, we will sort all images into Pre-Sorted Directories.
1 directory for all Images that directly share our Target Aspect-Ratio; `*/images/trainging-set/Aspect-Ratio`
and 1 more directory for all Images that do not share our Target Resolution or Aspect-Ratio; `*/images/training-set/Unmatched`

#### [1.1-1] Suggested Project Directory Schema

So first we should create our 5 directories in the projects root directory.
The Directory Schema should look something like this:

```sql
Root*/
|-- images/
|-- |-- NO/                             # All the images the model has decided was a '0.0 / NO' result.
|-- |-- RAW/                            # All the images
|-- |-- |-- (img#.png continued)
|-- |-- YES/                            # All the images the model has decided was a '1.0 / YES' result.
|-- |-- |-- (img#.%<ext>s continued)
|-- |-- training-set/                   # Directories of Pre-Sorted Images
|-- |-- |-- Aspect-Ratio/               # Images sharing the same Aspect-Ratio
|-- |-- |-- |-- (img#.png continued)
|-- |-- |-- NO/                         # Images that should result in a '0.0 / NO' response. Used for Pre-Training.
|-- |-- |-- |-- (img#.png continued)
|-- |-- |-- YES/                        # Images that should result in a `1.0 / YES` response. Used for Pre-Training.
|-- |-- |-- |-- (img#.png continued)
|-- |-- |-- Unmatched/                  # Images that do not share the Targets Aspect-Ratio. Used for Image-Resizing.
|-- |-- |-- |-- (img#.png continued)
|-- xAIx.ipynb                          # This Jupyter Notebook
```

#### [1.1-2] Valid Image Encoding Formats

All images encoding formats needs to be both `Read` & `Write` supported by Pythons `PILLOW` library. As of this time, the currently supported encoding formats are:

```json
{[
  {"AVIF": [".avif"]},
  {"BMP": [".bmp"]},
  {"DDS": [".dds"]},
  {"DIB": [".dib"]},
  {"EPS": [".eps"]},
  {"GIF": [".gif"]},
  {"ICNS": [".icns"]},
  {"ICO": [".ico"]},
  {"JPEG": [".jfif", ".jpe", ".jpeg", ".jpg"]},
  {"MPO": [".mpo"]},
  {"PCX": [".pcx"]},
  {"PNG": [".png"]},
  {"PBM": [".pbm"]},
  {"PGM": [".pgm"]},
  {"PPM": [".ppm"]},
  {"QOI": [".qoi"]},
  {"SGI": [".sgi"]},
  {"TGA": [".tga"]},
  {"TIFF": [".tif", ".tiff"]},
  {"WebP": [".webp"]},
  {"XBM": [".xbm"]},
  {"BLP": [".blp"]},
  {"IM": [".im"]},
  {"MSP": [".msp"]},
  {"SPIDER": [".spi"]}
]}
```

#### [1.1-3] Pre-Sorting Images into the same Encoding Format

To make things easier for us, we should first ensure that all our raw images are in the same encoding format.

```python
#!/use/bin/env python3

from pathlib import Path
from PIL import Image

TARGET_ENCODE = '.png'
TARGET_FORMAT = 'PNG'
IMAGES_DIR = Path(f"{Path.cwd()}/images/RAW")

valid = [
    ".avif", ".bmp", ".dds", ".dib",
    ".eps", ".gif", ".icns", ".ico",
    ".jfif", ".jpe", ".jpeg", ".jpg",
    ".mpo", ".pcx", ".png", ".pbm",
    ".pgm", ".ppm", ".qoi", ".sgi",
    ".tga", ".tif", ".tiff", ".webp",
    ".xbm", ".blp", ".im", ".msp",
    ".spi"
]

# Loop through all items in the directory
for item in IMAGES_DIR.iterdir():
    # Ensure its a file
    if not item.is_file():
        continue
      
    # Ensure its a valid image file
    if item.suffix.lower() not in valid:
        continue
    
    # Skip files already matching target encoding
    if item.suffix.lower() == TARGET_ENCODE:
        continue
  
    # Open, Convert, Save, Delete
    try:
        with Image.open(item) as image:
            image = image.convert("RGB")
            image.save(IMAGES_DIR, format=TARGET_FORMAT)
        item.unlink()
    except Exception as e:
        print(f"Error: {e}")
```

The above Py-Scripts Workflow:

```markdown
1. Target the `*/images/RAW` directory and loop through every item within.
2. It then makes sure the current item is actually a file,
  - If item `IS NOT` a file, skip this iteration
  - If item `IS` a file, continue to the next step of this iteration
3. Then it makes sure the item is a valid image file by comparing its extension to the list of valid extensions.
  - If item `IS NOT` a valid image format, skip this iteration
  - If item `IS` a valid image format, continue to the next step of this iteration
4. Checks if the current item is already in the target encoding format
  - If item `IS` the same encoding format, skip this iteration
  - If item `IS NOT` the same encoding format, continue to the next step of this iteration
5. Open the image into raw RGB format.
7. Save the image as the target encoding format back into the `*/images/RAW` directory
8. Deletes the original file that was not in the same encoding format
```

#### [1.1-4] Pre-Sorting Images that Fit into our Target Aspect-Ratio

Since AI expects the same sized and shaped inputs, we need to select what size our model intends to process.
Then as we process the images, we can take our target input resolution, and calculate its Aspect-Ratio.
Now images that share this Aspect-Ratio can be easily resized if its smaller than target, or multi-processed if its larger.

```python
#!/usr/bin/env python3

from pathlib import Path
from PIL import Image
from math import gcd

IMAGES_DIR = Path(f"{Path.cwd()}/images/RAW")
TARGET_DIR = Path(f"{Path.cwd()}/images/training-set/Aspect-Ratio")

# TARGET_RESOLUTION = None # My most common resolution: (832, 1248)

def calc_aspr(resolution: tuple[int, int]) -> tuple[int, int] | None:
    """1. Calculate and return an Images Aspect Ratio"""
    try:
        divisor = gcd(resolution[0], resolution[1])
        return tuple(int(resolution[0]  // divisor), int(resolution[1] // divisor))
    except Exception as e:
        print(f"Error: {e}")
        return None

def loop_raw(target: str) -> dict, dict | None:
    """1. Loop through the IMAGES_DIR and print each images Size and Aspect-Ratio.
    2. Counts how many images share the same Size and the same Aspect-Ratio.
    3. Returns the most common Resolution and Aspect-Ratio"""
    sizes = {}
    ratios = {}
    for item in IMAGES_DIR.iterdir():
        if not item.is_file():
            continue
        try:
            with Image.open(item) as image:
                name = image.name
                resolution = image.size
                aspr = str(calc_aspr(resolution))
                key = str(f"{resolution[0]}X{resolution[1]}")
                if key not in sizes.keys():
                    sizes[key] = [1,aspr]
                    ratios[aspr] = 1
                if key in sizes.keys():
                    sizes[key][0] += 1
                    ratios[aspr] += 1
            print(f"Image Details:\n  Name: {name},\n  Size: {resolution},\n  Aspect-Ratio: {aspr}")
            print(f"Totals:\n  Sizes: {sizes}\n  Aspect-Ratios: {ratios}")
        except Exception as e:
            print(f"Error: {e}")
            return None
        if sizes and ratios:

```

### [1.2] 

---

In [ ]:
#!/use/bin/env python3

## Ensures all photos are the same encoding type ##

from pathlib import Path
from PIL import Image

TARGET_ENCODE = '.png'
TARGET_FORMAT = 'PNG'
IMAGES_DIR = Path(f"{Path.cwd()}/images/RAW")

valid = [
    ".avif", ".bmp", ".dds", ".dib",
    ".eps", ".gif", ".icns", ".ico",
    ".jfif", ".jpe", ".jpeg", ".jpg",
    ".mpo", ".pcx", ".png", ".pbm",
    ".pgm", ".ppm", ".qoi", ".sgi",
    ".tga", ".tif", ".tiff", ".webp",
    ".xbm", ".blp", ".im", ".msp",
    ".spi"
]

# Loop through all items in the directory
for item in IMAGES_DIR.iterdir():
    # Ensure its a file
    if not item.is_file():
        continue
      
    # Ensure its a valid image file
    if item.suffix.lower() not in valid:
        continue
    
    # Skip files already matching target encoding
    if item.suffix.lower() == TARGET_ENCODE:
        continue
  
    # Open, Convert, Save, Delete
    try:
        with Image.open(item) as image:
            image = image.convert("RGB")
            image.save(IMAGES_DIR, format=TARGET_FORMAT)
        item.unlink()
    except Exception as e:
        print(f"Error: {e}")